# Fig 2: TTFT vs Input Length (Cached vs Uncached)

**Output:** `model_outputs/<MODEL_SHORT>/paper/section3/fig2/prefill_linearity.{pdf,png}`, `prefill_with_cache.{pdf,png}`

Paths are resolved via `MODEL_DATA_DIR` from `config/config.py` (set `MODELS_ROOT` in `config.env` to override).

### Call order
1. Auto-generate any missing long-prompt files (`model_outputs/<MODEL_SHORT>/long_prompts/`)
2. `profiling/launch_prefill_profile.sh` — collect cold-prefill data (~30–60 min, GPUs 1–3; writes to `model_outputs/<MODEL_SHORT>/paper/section3/profiling/prefill_profile_data/`)
3. `profiling/run_cache_cost.py` — collect cache-hit/miss data (writes to `model_outputs/<MODEL_SHORT>/paper/section3/profiling/cache_cost_data/`)
4. `scripts/plot_cache_cost.py` — parse cache data → `cache_cost_table.csv`
5. `scripts/plot_prefill_linearity.py` — cold-prefill scatter + quadratic fit
6. `scripts/plot_prefill_with_cache.py` — overlays cache-hit curve

Steps 2–3 collect raw data and only need to be run once. If data is already present the collect cells are safe to re-run (they skip existing output).

In [2]:
import subprocess
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd()] + list(Path.cwd().parents)
    if (p / ".conserve_root").exists()
) 

def run(cmd):
    buf = []
    with subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            buf.append(line)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {' '.join(str(c) for c in cmd)}")
    return "".join(buf)


In [5]:
# Ensure all required long-prompt files exist; generate any that are missing.
import sys
sys.path.insert(0, str(REPO_ROOT / "config"))
from config import MODEL_DATA_DIR, MODEL_SHORT

REQUIRED_LENGTHS = [
    1, 2, 4, 8, 16, 32, 64,
    128, 256, 512, 1024, 2048, 4096, 6144,
    10240, 12288, 16384, 20480, 24576, 28672, 32768,
    65536,
]
prompt_dir = MODEL_DATA_DIR / "long_prompts"
missing = [L for L in REQUIRED_LENGTHS if not (prompt_dir / f"prompts_{L}x2048.json").exists()]

if missing:
    print(f"[{MODEL_SHORT}] generating prompt files for L = {missing}")
    run(["python", str(REPO_ROOT / "profiling/long_prompts/generate_long_prompts.py"),
         "--targets", ",".join(str(L) for L in missing)])
else:
    print(f"[{MODEL_SHORT}] all prompt files present, skipping generation")


[Qwen3.6-27B] generating prompt files for L = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 6144, 10240, 12288, 16384, 20480, 24576, 28672, 32768, 65536]
[Qwen3.6-27B] copied seed → /data/projects/eicchen/conserve_project/conserve/model_outputs/Qwen3.6-27B/long_prompts/prompts_8192x2048.json
[Qwen3.6-27B] seed tokenized lengths — min=8286 max=8314 median=8299
[Qwen3.6-27B] generating L=1 (2048 prompts)…
  L=1: 0/2048
  L=1: 256/2048
  L=1: 512/2048
  L=1: 768/2048
  L=1: 1024/2048
  L=1: 1280/2048
  L=1: 1536/2048
  L=1: 1792/2048
  wrote prompts_1x2048.json: min=1 max=1 target=1  exact_match=2048/2048
[Qwen3.6-27B] generating L=2 (2048 prompts)…
  L=2: 0/2048
  L=2: 256/2048
  L=2: 512/2048
  L=2: 768/2048
  L=2: 1024/2048
  L=2: 1280/2048
  L=2: 1536/2048
  L=2: 1792/2048
  wrote prompts_2x2048.json: min=2 max=2 target=2  exact_match=2048/2048
[Qwen3.6-27B] generating L=4 (2048 prompts)…
  L=4: 0/2048
  L=4: 256/2048
  L=4: 512/2048
  L=4: 768/2048
  L=4: 1024/2048
  L=4:

In [3]:
# Steps 1-2: collect raw profiling data. Skip if data already exists.
run(["bash", str(REPO_ROOT / "profiling/launch_prefill_profile.sh")])

GPU 1  ←  L = 65536,28672,16384,6144,512
GPU 2  ←  L = 57344,32768,24576,10240,2048,128
GPU 3  ←  L = 49152,40960,20480,12288,8192,4096,1024,256

Waiting for shards...
  shard 0 (pid 3219470) OK
  shard 1 (pid 3219471) OK
  shard 2 (pid 3219472) OK
All shards done.


'GPU 1  ←  L = 65536,28672,16384,6144,512\nGPU 2  ←  L = 57344,32768,24576,10240,2048,128\nGPU 3  ←  L = 49152,40960,20480,12288,8192,4096,1024,256\n\nWaiting for shards...\n  shard 0 (pid 3219470) OK\n  shard 1 (pid 3219471) OK\n  shard 2 (pid 3219472) OK\nAll shards done.\n'

In [ ]:
run(["python", str(REPO_ROOT / "profiling/run_cache_cost.py")])

In [ ]:
# Steps 3-5: plot
%matplotlib inline
%run ../scripts/plot_cache_cost.py
%run ../scripts/plot_prefill_linearity.py
%run ../scripts/plot_prefill_with_cache.py